# Projeto final — Analisando dados históricos de ações e receitas
### Tesla (TSLA) e GameStop (GME) | Python, Pandas, yfinance e Web Scraping

Este notebook foi estruturado em **6 exercícios**, como na atividade do curso *Python Project for Data Science* (IBM/Coursera).

**Objetivo:** obter cotações históricas, extrair receitas trimestrais de páginas HTML, limpar os dados e apresentar dois painéis com preço das ações e receitas.

**Para executar:** use **Run → Run All Cells**, conectado à internet. Confirme que **todas as saídas e os dois gráficos** aparecem, salve o arquivo `.ipynb` e envie-o na plataforma. Os dados financeiros são recuperados na execução, não estão pré-preenchidos.

**Observação:** preço da ação e receita trimestral são séries com periodicidades diferentes. Visualizar tendências não prova causalidade nem constitui recomendação de investimento.

## Preparação do ambiente
A célula abaixo instala as bibliotecas somente se estiverem ausentes. A instalação inicial pode levar alguns minutos.

In [ ]:
import importlib.util
import subprocess
import sys

bibliotecas = {
    "yfinance": "yfinance",
    "pandas": "pandas",
    "requests": "requests",
    "bs4": "beautifulsoup4",
    "matplotlib": "matplotlib",
}

faltando = [pacote for modulo, pacote in bibliotecas.items()
            if importlib.util.find_spec(modulo) is None]
if faltando:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltando])
print("Bibliotecas preparadas.")

In [ ]:
import pandas as pd
import requests
import yfinance as yf
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup
from IPython.display import display

pd.set_option("display.max_columns", 15)
print("Importações concluídas.")

## Funções de apoio
A extração usa **BeautifulSoup** para localizar a tabela de receitas trimestrais na página HTML. Depois converte datas e remove os símbolos `$` e `,` dos valores de receita (em milhões de US$).

In [ ]:
def extrair_receita_trimestral(url, companhia):
    """Devolve DataFrame com colunas Date e Revenue (milhões de US$)."""
    resposta = requests.get(
        url,
        headers={"User-Agent": "Mozilla/5.0 (educational data analysis)"},
        timeout=40,
    )
    resposta.raise_for_status()
    sopa = BeautifulSoup(resposta.text, "html.parser")

    tabela = None
    termo = f"{companhia} Quarterly Revenue"
    for candidata in sopa.find_all("table"):
        if termo.lower() in candidata.get_text(" ", strip=True).lower():
            tabela = candidata
            break

    # Estrutura do material didático: primeira tabela = anual, segunda = trimestral.
    if tabela is None:
        tabelas_com_dados = [t for t in sopa.find_all("table") if t.find("tbody")]
        if len(tabelas_com_dados) >= 2:
            tabela = tabelas_com_dados[1]

    if tabela is None:
        raise ValueError(
            f"Tabela de receitas trimestrais não encontrada para {companhia}. "
            "Verifique se a página do material do curso continua acessível."
        )

    linhas = []
    corpo = tabela.find("tbody") or tabela
    for tr in corpo.find_all("tr"):
        colunas = tr.find_all("td")
        if len(colunas) < 2:
            continue
        data = colunas[0].get_text(" ", strip=True)
        receita = colunas[1].get_text(" ", strip=True)
        linhas.append((data, receita))

    receitas = pd.DataFrame(linhas, columns=["Date", "Revenue"])
    receitas["Revenue"] = (
        receitas["Revenue"].astype(str)
        .str.replace(r"[,$]", "", regex=True)
        .str.strip()
    )
    receitas["Revenue"] = pd.to_numeric(receitas["Revenue"], errors="coerce")
    receitas["Date"] = pd.to_datetime(receitas["Date"], errors="coerce")
    receitas.dropna(subset=["Date", "Revenue"], inplace=True)
    receitas["Date"] = receitas["Date"].dt.strftime("%Y-%m-%d")
    receitas.reset_index(drop=True, inplace=True)

    if receitas.empty:
        raise ValueError(f"Nenhuma receita válida extraída para {companhia}.")
    return receitas


def make_graph(stock_data, revenue_data, stock, data_final="2021-06-30"):
    """Desenha preço histórico e receita trimestral em dois gráficos."""
    acoes = stock_data.copy()
    receitas = revenue_data.copy()
    acoes["Date"] = pd.to_datetime(acoes["Date"], utc=True).dt.tz_localize(None)
    receitas["Date"] = pd.to_datetime(receitas["Date"])
    acoes = acoes.loc[acoes["Date"] <= pd.Timestamp(data_final)].sort_values("Date")
    receitas = receitas.loc[receitas["Date"] <= pd.Timestamp(data_final)].sort_values("Date")

    if acoes.empty or receitas.empty:
        raise ValueError(f"Não há dados suficientes até {data_final} para {stock}.")

    fig, eixos = plt.subplots(2, 1, figsize=(12, 8))
    eixos[0].plot(acoes["Date"], pd.to_numeric(acoes["Close"], errors="coerce"),
                  linewidth=1.4, color="#1767af")
    eixos[0].set_title(f"{stock} — Histórico do preço da ação")
    eixos[0].set_ylabel("Fechamento (US$)")

    eixos[1].plot(receitas["Date"], receitas["Revenue"],
                  linewidth=1.7, color="#21865a", marker=".", markersize=3)
    eixos[1].set_title(f"{stock} — Receita trimestral")
    eixos[1].set_ylabel("Receita (milhões de US$)")
    eixos[1].set_xlabel("Data")

    for eixo in eixos:
        eixo.grid(True, alpha=0.25)
        eixo.tick_params(axis="x", labelrotation=30)
    fig.tight_layout(pad=2.5)
    plt.show()
    return fig

print("Funções de extração e visualização definidas.")

## Exercício 1 — Extrair dados das ações da Tesla com yfinance
Cria o objeto Ticker `TSLA`, recupera o **maior histórico disponível**, redefine o índice para que `Date` seja uma coluna e mostra os cinco primeiros registros.

In [ ]:
tesla = yf.Ticker("TSLA")
tesla_data = tesla.history(period="max", auto_adjust=False)
if tesla_data.empty:
    raise ValueError("O Yahoo Finance não retornou cotações da Tesla. Tente novamente.")
tesla_data.reset_index(inplace=True)
display(tesla_data.head())

## Exercício 2 — Extrair receita trimestral da Tesla com Web Scraping
Fonte: página HTML usada no laboratório IBM. O DataFrame esperado é `tesla_revenue`, com colunas `Date` e `Revenue`.

In [ ]:
url_tesla = (
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
    "IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/revenue.htm"
)
tesla_revenue = extrair_receita_trimestral(url_tesla, "Tesla")
display(tesla_revenue.tail())

## Exercício 3 — Extrair dados das ações da GameStop com yfinance
Cria o objeto Ticker `GME`, recupera o maior histórico e mostra seus cinco primeiros registros.

In [ ]:
gamestop = yf.Ticker("GME")
gme_data = gamestop.history(period="max", auto_adjust=False)
if gme_data.empty:
    raise ValueError("O Yahoo Finance não retornou cotações da GameStop. Tente novamente.")
gme_data.reset_index(inplace=True)
display(gme_data.head())

## Exercício 4 — Extrair receita trimestral da GameStop com Web Scraping
Fonte: página HTML histórica usada no laboratório IBM. O DataFrame esperado é `gme_revenue`.

In [ ]:
url_gme = (
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
    "IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/stock.html"
)
gme_revenue = extrair_receita_trimestral(url_gme, "GameStop")
display(gme_revenue.tail())

## Exercício 5 — Painel Tesla: preço da ação × receita
Usa a função `make_graph()` para exibir os dois gráficos do material do curso, focando o período histórico até junho de 2021 para comparação com a atividade.

In [ ]:
fig_tesla = make_graph(tesla_data, tesla_revenue, "Tesla")

## Exercício 6 — Painel GameStop: preço da ação × receita
O gráfico de preço apresenta um salto expressivo próximo de 2021, que não é acompanhado por aumento proporcional das receitas no período disponível.

In [ ]:
fig_gme = make_graph(gme_data, gme_revenue, "GameStop")

## Validação final
Esta última célula confirma as quantidades de observações e as colunas essenciais de cada conjunto de dados.

In [ ]:
for nome, dados, colunas in [
    ("Ações Tesla", tesla_data, ["Date", "Close"]),
    ("Receita Tesla", tesla_revenue, ["Date", "Revenue"]),
    ("Ações GameStop", gme_data, ["Date", "Close"]),
    ("Receita GameStop", gme_revenue, ["Date", "Revenue"]),
]:
    assert all(coluna in dados.columns for coluna in colunas), f"Faltam colunas em {nome}"
    assert not dados.empty, f"Não há linhas em {nome}"
    print(f"{nome}: {len(dados):,} registros | colunas: {', '.join(colunas)}")
print("Validação concluída. Confira também se os dois painéis foram exibidos acima.")

---
### Referências e observações
- Curso IBM/Coursera: [Python Project for Data Science](https://www.coursera.org/learn/python-project-for-data-science).
- Bibliotecas: [yfinance](https://ranaroussi.github.io/yfinance/), [BeautifulSoup](https://www.crummy.com/software/BeautifulSoup/bs4/doc/), [Pandas](https://pandas.pydata.org/docs/), [Matplotlib](https://matplotlib.org/stable/).
- Fontes de receita: snapshots HTML educacionais do projeto IBM (`revenue.htm` e `stock.html`), reproduzindo dados históricos de Macrotrends.
- **Limite de verificação:** este arquivo contém o código completo, mas só fica pronto para submissão após ser executado com acesso à internet e ter todas as saídas salvas. Não há resultados de execução inventados.